# NS08 Tutorial C - Watts-Strogatz and the Small-World Mechanism

**Lecture:** NS08 - Network Models

**Reading:** Watts and Strogatz (1998)

## Learning goals
- Build the ring-lattice starting point explicitly.
- Understand how a small rewiring probability creates shortcuts.
- Measure the trade-off between clustering and path length as $p$ changes.
- Compare a real clustered network to ER and Watts-Strogatz benchmarks.


In [ ]:
from netsci_utils import *
import pandas as pd

set_seeds()
%matplotlib inline


def ring_lattice(n, k):
    if k % 2 != 0:
        raise ValueError('k must be even for a symmetric ring lattice')
    G = nx.Graph()
    G.add_nodes_from(range(n))
    for node in range(n):
        for step in range(1, k // 2 + 1):
            G.add_edge(node, (node + step) % n)
            G.add_edge(node, (node - step) % n)
    return G


def ws_sweep(n, k, p_values, seed=RANDOM_SEED):
    base = ring_lattice(n, k)
    C0 = nx.average_clustering(base)
    L0 = nx.average_shortest_path_length(base)
    rows = []
    for p in p_values:
        G = nx.watts_strogatz_graph(n, k, p, seed=seed)
        C = nx.average_clustering(G)
        L = nx.average_shortest_path_length(G)
        rows.append({
            'p': p,
            'C(p)': C,
            'L(p)': L,
            'C(p)/C(0)': C / C0,
            'L(p)/L(0)': L / L0,
        })
    return pd.DataFrame(rows)


def graph_summary(G, name):
    degrees = np.array([degree for _, degree in G.degree()], dtype=float)
    largest_nodes = max(nx.connected_components(G), key=len)
    largest = G.subgraph(largest_nodes).copy()
    return {
        'network': name,
        'n': G.number_of_nodes(),
        'm': G.number_of_edges(),
        '<k>': degrees.mean(),
        'max k': degrees.max(),
        'kappa': heterogeneity_kappa(G),
        'avg clustering': nx.average_clustering(G),
        'avg path length in LCC': nx.average_shortest_path_length(largest),
    }


def fit_ws_candidates(n, k, target_clustering, target_path, p_values, seed=RANDOM_SEED):
    rows = []
    for p in p_values:
        G = nx.watts_strogatz_graph(n, k, p, seed=seed)
        C = nx.average_clustering(G)
        L = nx.average_shortest_path_length(G)
        rows.append({
            'p': p,
            'avg clustering': C,
            'avg path length': L,
            'clustering gap': abs(C - target_clustering),
            'path gap': abs(L - target_path),
            'combined gap': abs(C - target_clustering) + abs(L - target_path),
        })
    return pd.DataFrame(rows)

---
## 1. A real network with clustering and short paths

We use the **Les Miserables** character co-occurrence network as the observed benchmark.
It is not hub-dominated like OpenFlights, but it is a good example of a network with strong local clustering and short global paths.


In [ ]:
lesmis = nx.les_miserables_graph()
avg_degree_target = np.mean([degree for _, degree in lesmis.degree()])
er_match = nx.erdos_renyi_graph(
    lesmis.number_of_nodes(),
    avg_degree_target / (lesmis.number_of_nodes() - 1),
    seed=RANDOM_SEED,
)

comparison_df = pd.DataFrame([
    graph_summary(lesmis, 'Les Miserables'),
    graph_summary(er_match, 'Matched ER'),
])
print(comparison_df.round(3).to_string(index=False))


**Interpretation.**
- ER gives short paths, but its clustering is far too low.
- This is the exact modelling gap that Watts-Strogatz tries to fill.


---
## 2. Start from a ring lattice

Watts-Strogatz begins from a highly structured graph: each node is connected to its $k$ nearest neighbours on a ring.
That construction creates strong local overlap and therefore high clustering.


In [ ]:
ring = ring_lattice(n=20, k=4)
ring_pos = nx.circular_layout(ring)
plot_graph(
    ring,
    pos=ring_pos,
    with_labels=True,
    node_size=500,
    title='Ring lattice with n = 20 and k = 4',
)

print_network_stats(ring)


---
## 3. A few shortcuts change path length dramatically

We sweep the rewiring probability $p$ and measure:
- clustering $C(p)$,
- average path length $L(p)$,
- their normalized versions relative to the ring lattice.


In [ ]:
p_values = np.array([0.0, 0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1.0])
sweep_df = ws_sweep(n=200, k=6, p_values=p_values)

fig, ax = plt.subplots(figsize=FIGURE_SIZE)
ax.plot(sweep_df['p'], sweep_df['C(p)/C(0)'], marker='o', linewidth=2, color=CATEGORY_PALETTE['blue'], label='Normalized clustering')
ax.plot(sweep_df['p'], sweep_df['L(p)/L(0)'], marker='o', linewidth=2, color=CATEGORY_PALETTE['orange'], label='Normalized path length')
style_axis(
    ax,
    title='Watts-Strogatz sweep: clustering versus path length',
    xlabel='Rewiring probability p',
    ylabel='Normalized metric',
    xscale='log',
    legend=True,
)
plt.show()

print(sweep_df.round(4).to_string(index=False))


**Interpretation.**
- For very small $p$, clustering stays close to the lattice value while path length drops sharply.
- That narrow region is the **small-world regime**.
- For large $p$, the graph approaches a random graph: short paths remain, but clustering collapses.


---
## 4. Three concrete examples of the rewiring mechanism

We compare the same model at three probabilities:
- $p = 0$: regular lattice,
- $p = 0.05$: small-world regime,
- $p = 1$: heavily randomized network.


In [ ]:
examples = {
    'p = 0.00': nx.watts_strogatz_graph(30, 4, 0.00, seed=RANDOM_SEED),
    'p = 0.05': nx.watts_strogatz_graph(30, 4, 0.05, seed=RANDOM_SEED),
    'p = 1.00': nx.watts_strogatz_graph(30, 4, 1.00, seed=RANDOM_SEED),
}
pos = nx.circular_layout(next(iter(examples.values())))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (title, graph) in zip(axes, examples.items()):
    draw_graph(graph, pos=pos, ax=ax, with_labels=False, node_size=180, width=1.0)
    ax.set_title(title)
plt.show()


---
## 5. Can Watts-Strogatz match the real network?

We try a grid of candidate $p$ values for a model with the same number of nodes and a similar average degree as Les Miserables.
This makes the modelling trade-off explicit rather than qualitative.


In [ ]:
n_ref = lesmis.number_of_nodes()
k_ref = 6
target_clustering = nx.average_clustering(lesmis)
target_path = nx.average_shortest_path_length(lesmis)
candidate_ps = np.array([0.01, 0.03, 0.05, 0.10, 0.20, 0.30])

fit_df = fit_ws_candidates(n_ref, k_ref, target_clustering, target_path, candidate_ps)
print(fit_df.round(3).to_string(index=False))

best_row = fit_df.loc[fit_df['combined gap'].idxmin()]
best_p = float(best_row['p'])
ws_best = nx.watts_strogatz_graph(n_ref, k_ref, best_p, seed=RANDOM_SEED)

model_compare_df = pd.DataFrame([
    graph_summary(lesmis, 'Les Miserables'),
    graph_summary(er_match, 'Matched ER'),
    graph_summary(ws_best, f'Best WS candidate (p = {best_p:.2f})'),
])
print('\nComparison to the real network:')
print(model_compare_df.round(3).to_string(index=False))


**Interpretation.**
- Watts-Strogatz improves on ER because it can retain clustering while shortening paths.
- But it still has a narrow degree distribution, so it cannot explain hub-dominated networks.
- This is why the lecture treats Watts-Strogatz as a mechanism for **shortcuts**, not for heterogeneity.


## Takeaway

- The Watts-Strogatz model addresses the main weakness of ER: low clustering.
- Its key idea is that a few long-range shortcuts can collapse path lengths without destroying local neighbourhood structure.
- It is therefore a model of the **small-world mechanism**, not a model of hub formation.
- That limitation is exactly what motivates the next lecture block on growth and preferential attachment.
